In [1]:
!pip install openai chromadb pypdf tiktoken python-dotenv

     --------------------------------------- 23.4/23.4 MB 26.1 MB/s eta 0:00:00
     ------------------------------------- 336.3/336.3 kB 20.4 MB/s eta 0:00:00
     ------------------------------------- 879.4/879.4 kB 28.0 MB/s eta 0:00:00
  Using cached pydantic_settings-2.13.1-py3-none-any.whl (58 kB)
     --------------------------------------- 12.6/12.6 MB 40.9 MB/s eta 0:00:00
     ---------------------------------------- 69.0/69.0 kB 3.7 MB/s eta 0:00:00
     ------------------------------------- 180.2/180.2 kB 11.3 MB/s eta 0:00:00
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
     ---------------------------------------- 60.6/60.6 kB 3.1 MB/s eta 0:00:00
     ---------------------------------------- 4.9/4.9 MB 44.4 MB/s eta 0:00:00
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl (150 kB)
  Using cached typer-0.24.1-py3-none-any.whl (56 kB)
     ---------------------------------------- 2.0/2.0 MB 42.7 MB/s eta 0:00:00
  Using cached pyyaml-6.0.3-cp311-cp3


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader

# -------------------------------
# Setup
# -------------------------------
import os

os.environ["OPENAI_API_KEY"] = ""
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------------
# Step 1: Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

# -------------------------------
# Step 2: Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Step 3: Create Embeddings
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Step 4: Store in ChromaDB
# -------------------------------
def create_vector_db(chunks):
    chroma_client = chromadb.Client()
    collection = chroma_client.create_collection(name="rag_demo")

    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)
        collection.add(
            ids=[str(i)],
            embeddings=[embedding],
            documents=[chunk]
        )

    return collection

# -------------------------------
# Step 5: Retrieve Relevant Docs
# -------------------------------
def retrieve(query, collection, top_k=3):
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]
        # Print retrieved chunks
    print("\nRetrieved Chunks:\n" + "-"*50)
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n{doc[:300]}...")  # show first 300 chars

    return docs

# -------------------------------
# Step 6: Generate Answer
# -------------------------------
def generate_answer(query, context_docs):
    context = "\n\n".join(context_docs)

    prompt = f"""
You are a helpful assistant.
Answer the question based ONLY on the context below.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content



In [5]:
# -------------------------------
# MAIN PIPELINE
# -------------------------------
if __name__ == "__main__":
    pdf_path = "HR Policy Manual 2025.pdf"

    print("📄 Loading PDF...")
    text = load_pdf(pdf_path)

    print("✂️ Chunking...")
    chunks = chunk_text(text)

    print("🧠 Creating Vector DB...")
    collection = create_vector_db(chunks)

    print("✅ Ready for questions!")



📄 Loading PDF...
✂️ Chunking...
🧠 Creating Vector DB...
✅ Ready for questions!


In [6]:
if __name__ == "__main__":
    while True:
          query = input("What is leave policy? ")
          if query.lower() == "exit":
              break

          docs = retrieve(query, collection)
          answer = generate_answer(query, docs)

          print("\n💡 Answer:")
          print(answer)


💡 Answer:
The leave policy outlined in the context includes the following key points:

1. **Termination of Appointment**: If an employee takes leave, their appointment at the Institute will be treated as terminated after a certain period.

2. **Limit on Leave**: Permanent employees cannot be granted leave for a continuous period exceeding five years. However, any amount of Earned Leave (EOL) may be sanctioned within this limitation.

3. **Maternity Leave**: 
   - Married/unmarried female employees are entitled to 180 days of maternity leave during pregnancy.
   - Maternity leave is only admissible to employees with less than two surviving children.

4. **Procedure for Granting Leave**:
   - The grant of leave is governed by the Institute Leave Rules, which align with the leave rules applicable to Central Government employees.
   - Leave cannot be claimed as a right and may be denied based on the Institute’s requirements or public exigencies.
   - The leave sanctioning authority has th

In [9]:
!pip install google

     ---------------------------------------- 45.3/45.3 kB 1.1 MB/s eta 0:00:00
     -------------------------------------- 107.7/107.7 kB 2.1 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ================================
# SIMPLE RAG IN ONE CELL (COLAB)
# ================================

# Install dependencies
# !pip install -q openai chromadb pypdf tiktoken

# -------------------------------
# Setup
# -------------------------------
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader
from google.colab import files

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

# -------------------------------
# Upload PDF
# -------------------------------
print("Upload your PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# -------------------------------
# Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

# -------------------------------
# Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Embedding
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Create Vector DB
# -------------------------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

# -------------------------------
# Build Index
# -------------------------------
print("Loading PDF...")
text = load_pdf(pdf_path)

print("Chunking...")
chunks = chunk_text(text)

print("Creating embeddings and storing...")
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk]
    )

print("RAG Ready")

# -------------------------------
# Retriever
# -------------------------------
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    return results["documents"][0]

# -------------------------------
# Generator (LLM)
# -------------------------------
def generate_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

# -------------------------------
# Ask Questions Loop
# -------------------------------
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    ans = generate_answer(q, docs)

    print("Answer:")
    print(ans)

Upload your PDF


NameError: name 'files' is not defined

In [ ]:
# ================================
# SIMPLE RAG WITH STREAMING (COLAB)
# ================================

# Install dependencies
!pip install -q openai chromadb pypdf tiktoken

# -------------------------------
# Setup
# -------------------------------
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader
from google.colab import files

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"
client = OpenAI()

# -------------------------------
# Upload PDF
# -------------------------------
print("Upload your PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# -------------------------------
# Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

# -------------------------------
# Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Embedding
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Create Vector DB
# -------------------------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

# -------------------------------
# Build Index
# -------------------------------
print("Loading PDF...")
text = load_pdf(pdf_path)

print("Chunking...")
chunks = chunk_text(text)

print("Creating embeddings and storing...")
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk]
    )

print("RAG Ready")

# -------------------------------
# Retriever
# -------------------------------
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    return results["documents"][0]

# -------------------------------
# Streaming Generator
# -------------------------------
def stream_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        stream=True
    )

    full_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            print(token, end="", flush=True)
            full_text += token

    print()  # newline after streaming
    return full_text

# -------------------------------
# Ask Questions Loop
# -------------------------------
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    print("Answer:")
    stream_answer(q, docs)